In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "Develepor_X":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import Rock_Genetics.rock_genetic_helper as genetics

from Rock_AI.datasets.breeding_dataset_generator import BreedingDatasetGenerator
from Rock_GameState.rock_game_state_helper import GameMaster


In [2]:

game = GameMaster(seed=123)

parent_a = next(
    rock for rock in game.rocks.values()
    if rock.sex == genetics.Sex.MALE
)
parent_b = next(
    rock for rock in game.rocks.values()
    if rock.sex == genetics.Sex.FEMALE
)

generator = BreedingDatasetGenerator(seed=5000)
records = generator.generate_from_parent_pairs(
    [(parent_a, parent_b)],
    trials_per_pair=10,
)

# IF SAVING THE DATA!
#generator.write_jsonl(records, "training_data/breeding_records.jsonl")

In [3]:
from Rock_AI.evaluation.pair_evaluator import PairEvaluator, PairUtilityWeights
from Rock_GameState.rock_game_state_helper import GameMaster

# A GameMaster currently serves as the small player farm.
farm = GameMaster(seed=79)

evaluator = PairEvaluator()
weights = PairUtilityWeights(
    expected_value_weight=1.0,
    maximum_value_weight=0.25,
    survivor_count_weight=0.5,
    genotype_diversity_weight=2.0,
    phenotype_diversity_weight=2.0,
    rare_trait_weight=3.0,
    mutation_opportunity_weight=0.5,
)

# Invalid pairs are excluded using BreedingMaster.validate_breeding_pair().
ranked_pairs = evaluator.rank_pairs(
    farm,
    trial_count=1000,
    seed=900,
    weights=weights,
    game=farm,
)

print(f"Legal pairs evaluated: {len(ranked_pairs)}")

top = ranked_pairs[0]
expectation = top.expectation

print(f"Top pair: {top.parent_ids}")
print(f"Utility score: {top.combined_utility_score:.3f}")
print("Score breakdown:", top.score_components)
print("Expected child value:", expectation.expected_child_value.mean)
print("Expected maximum value:", expectation.expected_maximum_child_value.mean)
print("Expected raw clutch:", expectation.expected_raw_clutch_size.mean)
print("Expected survivors:", expectation.expected_survivor_count.mean)
print("Expected mutations per child:", expectation.expected_mutations_per_child)
print("95% value interval:", expectation.expected_child_value.confidence_95)

Legal pairs evaluated: 4
Top pair: (3, 4)
Utility score: 16.759
Score breakdown: {'expected_value': 3.9720500000000003, 'maximum_value': 1.55925, 'survivor_value': 6.577419215981278, 'genotype_diversity': 2.0, 'phenotype_diversity': 1.9720380952380951, 'rare_trait': 0.19851562499999986, 'mutation_opportunity': 0.47941679294094525}
Expected child value: 3.9720500000000003
Expected maximum value: 6.237
Expected raw clutch: 3.511
Expected survivors: 3.341
Expected mutations per child: 0.54
95% value interval: (3.886148281730354, 4.0579517182696465)


In [4]:
runtime = AgentRuntimeManager()
session = runtime.create_session(
    agent=neural_agent,
    environment=environment,
    seed=1234,
    objective_profile=balanced_profile,
)

runtime.apply(session.session_id, StartSessionCommand())
result = runtime.apply(session.session_id, StepSessionCommand())

print(result.event.summary)
print(result.decision_explanation)

runtime.apply(session.session_id, PauseSessionCommand())
runtime.save_session(session.session_id, "runtime_sessions/farmer_001.json")

NameError: name 'AgentRuntimeManager' is not defined